# Eksplorasi SQL - Olist E-Commerce

Eksplorasi awal data menggunakan SQL untuk mengidentifikasi masalah kualitas data
yang perlu ditangani saat proses cleaning.

**Prasyarat:** Jalankan `01_load_to_sqlite.ipynb` terlebih dahulu.

## Persiapan

In [29]:
import sqlite3

import pandas as pd

In [30]:
DB_PATH = "../data/database/olist.db"
conn = sqlite3.connect(DB_PATH)

---
## 1. Pratinjau Data

Melihat struktur dan contoh nilai dari setiap tabel.

In [31]:
pd.read_sql_query("SELECT * FROM customers LIMIT 5", conn)

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [32]:
pd.read_sql_query("SELECT * FROM orders LIMIT 5", conn)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [33]:
pd.read_sql_query("SELECT * FROM order_item LIMIT 5", conn)

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [34]:
pd.read_sql_query("SELECT * FROM products LIMIT 5", conn)

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


---
## 2. Jumlah Baris

Mengetahui skala data di setiap tabel.

In [35]:
pd.read_sql_query("""
SELECT 'customers' AS tabel, COUNT(*) AS jumlah_baris FROM customers
UNION ALL
SELECT 'orders', COUNT(*) FROM orders
UNION ALL
SELECT 'order_item', COUNT(*) FROM order_item
UNION ALL
SELECT 'products', COUNT(*) FROM products
""", conn)

,tabel,jumlah_baris
0,customers,99441
1,orders,99441
2,order_item,112650
3,products,32951


---
## 3. Nilai Kosong (Missing Values)

Menghitung jumlah NULL per kolom untuk menentukan kolom mana yang perlu ditangani.

### 3a. orders

In [36]:
pd.read_sql_query("""
SELECT
    SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) AS null_order_id,
    SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) AS null_customer_id,
    SUM(CASE WHEN order_status IS NULL THEN 1 ELSE 0 END) AS null_order_status,
    SUM(CASE WHEN order_purchase_timestamp IS NULL THEN 1 ELSE 0 END) AS null_purchase_ts,
    SUM(CASE WHEN order_approved_at IS NULL THEN 1 ELSE 0 END) AS null_approved_at,
    SUM(CASE WHEN order_delivered_carrier_date IS NULL THEN 1 ELSE 0 END) AS null_carrier_date,
    SUM(CASE WHEN order_delivered_customer_date IS NULL THEN 1 ELSE 0 END) AS null_customer_date,
    SUM(CASE WHEN order_estimated_delivery_date IS NULL THEN 1 ELSE 0 END) AS null_estimated_date
FROM orders
""", conn)

,null_order_id,null_customer_id,null_order_status,null_purchase_ts,null_approved_at,null_carrier_date,null_customer_date,null_estimated_date
0,0,0,0,0,160,1783,2965,0


### 3b. order_item

In [37]:
pd.read_sql_query("""
SELECT
    SUM(CASE WHEN order_item_id IS NULL THEN 1 ELSE 0 END) AS null_order_item_id,
    SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) AS null_order_id,
    SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END) AS null_product_id,
    SUM(CASE WHEN seller_id IS NULL THEN 1 ELSE 0 END) AS null_seller_id,
    SUM(CASE WHEN price IS NULL THEN 1 ELSE 0 END) AS null_price,
    SUM(CASE WHEN freight_value IS NULL THEN 1 ELSE 0 END) AS null_freight_value,
    SUM(CASE WHEN shipping_limit_date IS NULL THEN 1 ELSE 0 END) AS null_shipping_limit
FROM order_item
""", conn)

,null_order_item_id,null_order_id,null_product_id,null_seller_id,null_price,null_freight_value,null_shipping_limit
0,0,0,0,0,0,0,0


### 3c. products

In [38]:
pd.read_sql_query("""
SELECT
    SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END) AS null_product_id,
    SUM(CASE WHEN product_category_name IS NULL THEN 1 ELSE 0 END) AS null_category,
    SUM(CASE WHEN product_name_lenght IS NULL THEN 1 ELSE 0 END) AS null_name_len,
    SUM(CASE WHEN product_description_lenght IS NULL THEN 1 ELSE 0 END) AS null_desc_len,
    SUM(CASE WHEN product_photos_qty IS NULL THEN 1 ELSE 0 END) AS null_photos_qty,
    SUM(CASE WHEN product_weight_g IS NULL THEN 1 ELSE 0 END) AS null_weight,
    SUM(CASE WHEN product_length_cm IS NULL THEN 1 ELSE 0 END) AS null_length,
    SUM(CASE WHEN product_height_cm IS NULL THEN 1 ELSE 0 END) AS null_height,
    SUM(CASE WHEN product_width_cm IS NULL THEN 1 ELSE 0 END) AS null_width
FROM products
""", conn)

,null_product_id,null_category,null_name_len,null_desc_len,null_photos_qty,null_weight,null_length,null_height,null_width
0,0,610,610,610,610,2,2,2,2


### 3d. customers

In [39]:
pd.read_sql_query("""
SELECT
    SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) AS null_customer_id,
    SUM(CASE WHEN customer_unique_id IS NULL THEN 1 ELSE 0 END) AS null_unique_id,
    SUM(CASE WHEN customer_zip_code_prefix IS NULL THEN 1 ELSE 0 END) AS null_zip,
    SUM(CASE WHEN customer_city IS NULL THEN 1 ELSE 0 END) AS null_city,
    SUM(CASE WHEN customer_state IS NULL THEN 1 ELSE 0 END) AS null_state
FROM customers
""", conn)

,null_customer_id,null_unique_id,null_zip,null_city,null_state
0,0,0,0,0,0


---
## 4. Cek Duplikat

Memastikan primary key benar-benar unik di setiap tabel.

In [40]:
# order_id harus unik di tabel orders
df = pd.read_sql_query("""
SELECT order_id, COUNT(*) AS jumlah
FROM orders
GROUP BY order_id
HAVING COUNT(*) > 1
""", conn)
print(f"Duplikat order_id di orders: {len(df)}")
df.head()

Duplikat order_id di orders: 0


,order_id,jumlah


In [41]:
# customer_id harus unik di tabel customers
df = pd.read_sql_query("""
SELECT customer_id, COUNT(*) AS jumlah
FROM customers
GROUP BY customer_id
HAVING COUNT(*) > 1
""", conn)
print(f"Duplikat customer_id di customers: {len(df)}")
df.head()

Duplikat customer_id di customers: 0


,customer_id,jumlah


In [42]:
# product_id harus unik di tabel products
df = pd.read_sql_query("""
SELECT product_id, COUNT(*) AS jumlah
FROM products
GROUP BY product_id
HAVING COUNT(*) > 1
""", conn)
print(f"Duplikat product_id di products: {len(df)}")
df.head()

Duplikat product_id di products: 0


,product_id,jumlah


In [43]:
# (order_id, order_item_id) harus unik di tabel order_item
df = pd.read_sql_query("""
SELECT order_id, order_item_id, COUNT(*) AS jumlah
FROM order_item
GROUP BY order_id, order_item_id
HAVING COUNT(*) > 1
""", conn)
print(f"Duplikat (order_id, order_item_id) di order_item: {len(df)}")
df.head()

Duplikat (order_id, order_item_id) di order_item: 0


,order_id,order_item_id,jumlah


---
## 5. Rentang Nilai dan Outlier

Mendeteksi nilai anomali: negatif, nol, atau terlalu besar.

In [44]:
# Harga dan ongkos kirim di order_item
pd.read_sql_query("""
SELECT
    MIN(price) AS harga_min,
    MAX(price) AS harga_maks,
    ROUND(AVG(price), 2) AS harga_rata,
    SUM(CASE WHEN price <= 0 THEN 1 ELSE 0 END) AS harga_nol_negatif,
    MIN(freight_value) AS ongkir_min,
    MAX(freight_value) AS ongkir_maks,
    ROUND(AVG(freight_value), 2) AS ongkir_rata,
    SUM(CASE WHEN freight_value < 0 THEN 1 ELSE 0 END) AS ongkir_negatif
FROM order_item
""", conn)

,harga_min,harga_maks,harga_rata,harga_nol_negatif,ongkir_min,ongkir_maks,ongkir_rata,ongkir_negatif
0,0.85,6735.0,120.65,0,0.0,409.68,19.99,0


In [45]:
# Dimensi dan berat produk
pd.read_sql_query("""
SELECT
    MIN(product_weight_g) AS berat_min,    MAX(product_weight_g) AS berat_maks,
    MIN(product_length_cm) AS panjang_min,  MAX(product_length_cm) AS panjang_maks,
    MIN(product_height_cm) AS tinggi_min,   MAX(product_height_cm) AS tinggi_maks,
    MIN(product_width_cm) AS lebar_min,     MAX(product_width_cm) AS lebar_maks,
    SUM(CASE WHEN product_weight_g <= 0 THEN 1 ELSE 0 END) AS berat_nol_negatif,
    SUM(CASE WHEN product_length_cm <= 0 THEN 1 ELSE 0 END) AS panjang_nol_negatif
FROM products
""", conn)

,berat_min,berat_maks,panjang_min,panjang_maks,tinggi_min,tinggi_maks,lebar_min,lebar_maks,berat_nol_negatif,panjang_nol_negatif
0,0.0,40425.0,7.0,105.0,2.0,105.0,6.0,118.0,4,0


---
## 6. Validasi Tanggal

Memeriksa rentang tanggal dan memastikan urutan kronologis yang logis:
`purchase < approved < carrier < delivered`.

In [46]:
# Rentang tanggal order
pd.read_sql_query("""
SELECT
    MIN(order_purchase_timestamp) AS order_pertama,
    MAX(order_purchase_timestamp) AS order_terakhir
FROM orders
""", conn)

,order_pertama,order_terakhir
0,2016-09-04 21:15:19,2018-10-17 17:30:18


In [47]:
# Order dengan urutan tanggal yang tidak logis
pd.read_sql_query("""
SELECT
    SUM(CASE WHEN order_approved_at < order_purchase_timestamp THEN 1 ELSE 0 END)
        AS approved_sebelum_purchase,
    SUM(CASE WHEN order_delivered_carrier_date < order_approved_at THEN 1 ELSE 0 END)
        AS carrier_sebelum_approved,
    SUM(CASE WHEN order_delivered_customer_date < order_delivered_carrier_date THEN 1 ELSE 0 END)
        AS delivered_sebelum_carrier
FROM orders
WHERE order_approved_at IS NOT NULL
  AND order_delivered_carrier_date IS NOT NULL
  AND order_delivered_customer_date IS NOT NULL
""", conn)

,approved_sebelum_purchase,carrier_sebelum_approved,delivered_sebelum_carrier
0,0,1350,23


---
## 7. Distribusi Kategorikal

Memeriksa inkonsistensi, typo, atau nilai yang tidak terduga pada kolom kategorikal.

In [48]:
# Distribusi status order
pd.read_sql_query("""
SELECT
    order_status,
    COUNT(*) AS jumlah,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM orders), 2) AS persentase
FROM orders
GROUP BY order_status
ORDER BY jumlah DESC
""", conn)

,order_status,jumlah,persentase
0,delivered,96478,97.02
1,shipped,1107,1.11
2,canceled,625,0.63
3,unavailable,609,0.61
4,invoiced,314,0.32
5,processing,301,0.30
6,created,5,0.01
7,approved,2,0.00


In [49]:
# Distribusi kategori produk
pd.read_sql_query("""
SELECT product_category_name, COUNT(*) AS jumlah
FROM products
WHERE product_category_name IS NOT NULL
GROUP BY product_category_name
ORDER BY product_category_name
""", conn)

,product_category_name,jumlah
0,agro_industria_e_comercio,74
1,alimentos,82
2,alimentos_bebidas,104
3,artes,55
4,artes_e_artesanato,19
...,...,...
68,sinalizacao_e_seguranca,93
69,tablets_impressao_imagem,9
70,telefonia,1134
71,telefonia_fixa,116


In [50]:
# 30 kota customer terbanyak (cek inkonsistensi ejaan)
pd.read_sql_query("""
SELECT customer_city, COUNT(*) AS jumlah
FROM customers
GROUP BY customer_city
ORDER BY jumlah DESC
LIMIT 30
""", conn)

,customer_city,jumlah
0,sao paulo,15540
1,rio de janeiro,6882
2,belo horizonte,2773
3,brasilia,2131
4,curitiba,1521
5,campinas,1444
6,porto alegre,1379
7,salvador,1245
8,guarulhos,1189
9,sao bernardo do campo,938


---
## 8. Integritas Relasi Antar Tabel

Memastikan semua foreign key merujuk ke record yang valid.

In [51]:
# Baris order_item yang tidak punya pasangan di tabel orders
pd.read_sql_query("""
SELECT COUNT(*) AS order_item_tanpa_order
FROM order_item oi
    LEFT JOIN orders o ON oi.order_id = o.order_id
WHERE o.order_id IS NULL
""", conn)

,order_item_tanpa_order
0,0


In [52]:
# Baris order_item yang tidak punya pasangan di tabel products
pd.read_sql_query("""
SELECT COUNT(*) AS order_item_tanpa_produk
FROM order_item oi
    LEFT JOIN products p ON oi.product_id = p.product_id
WHERE p.product_id IS NULL
""", conn)

,order_item_tanpa_produk
0,0


In [53]:
# Baris orders yang tidak punya pasangan di tabel customers
pd.read_sql_query("""
SELECT COUNT(*) AS order_tanpa_customer
FROM orders o
    LEFT JOIN customers c ON o.customer_id = c.customer_id
WHERE c.customer_id IS NULL
""", conn)

,order_tanpa_customer
0,0


---
## Ringkasan

Temuan utama dari eksplorasi ini yang menentukan langkah cleaning di notebook selanjutnya.

| Area | Temuan | Tindakan |
|------|--------|----------|
| Nilai kosong | `order_approved_at` (160), `order_delivered_carrier_date` (1.783), `order_delivered_customer_date` (2.965) memiliki NULL signifikan. Di tabel products, `product_category_name` (610), `product_name_lenght` (610), dan `product_weight_g` (2) juga NULL. | Evaluasi apakah NULL wajar berdasarkan status order (misal: belum dikirim). Untuk produk, pertimbangkan imputasi atau penghapusan baris. |
| Duplikat | Tidak ditemukan duplikat pada primary key manapun: `order_id`, `customer_id`, `product_id`, dan `(order_id, order_item_id)`. | Tidak perlu tindakan. |
| Outlier numerik | Harga: 0,85 - 6.735,00 (rata-rata 120,65). Freight: 0,00 - 409,68. Tidak ada nilai negatif. Berat produk: 0 - 40.425 g. Panjang: 7 - 105 cm. | Investigasi berat 0 g. Periksa harga dan dimensi ekstrem apakah merupakan outlier nyata. |
| Anomali tanggal | Rentang data: Sep 2016 - Okt 2018. Ditemukan 1.350 kasus carrier sebelum approved, dan 23 kasus delivered sebelum carrier. Tidak ada approved sebelum purchase. | Investigasi 1.350 kasus anomali urutan tanggal. Kemungkinan kesalahan pencatatan atau perbedaan zona waktu. |
| Kategorikal | 8 jenis status order; delivered mendominasi (96.478 dari 99.441). Terdapat 73 kategori produk. | Periksa inkonsistensi ejaan pada nama kota dan kategori produk. |
| Integritas relasi | Semua foreign key valid: tidak ada orphan record di ketiga relasi yang dicek. | Tidak perlu tindakan. |

In [54]:
conn.close()